# 005 — Error Analysis on Labeled Soundscapes

Analyze exp 012 model predictions on 66 labeled soundscape files.

**Prereq**: Run inference first:
```bash
python scripts/infer_soundscapes.py --model all --split labeled
```

In [1]:
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

DATA_DIR = Path("/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026")
INFERENCE_DIR = Path("/storage/project/ps-clef2026_birdclef-0/shared/inference/birdclef-2026")
WEIGHTS_DIR = Path(
    "/storage/project/ps-clef2026_birdclef-0/mgustineli3/birdclef-2026"
    "/user/murilo/experiments/010-submission-pipeline/artifacts/weights"
)

MODELS = {
    "ProtoCLR": "protoclr-2026-soundscape",
    "ConvNeXt": "convnext-2026-soundscape",
    "EAT": "eat-2026-soundscape",
}

## Load ground truth + taxonomy

In [2]:
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
all_species = sorted(taxonomy["primary_label"].astype(str).tolist())
sp2idx = {sp: i for i, sp in enumerate(all_species)}
sp2taxon = dict(zip(taxonomy["primary_label"].astype(str), taxonomy["class_name"], strict=False))
print(f"Taxonomy: {len(all_species)} species")
print(f"  {taxonomy['class_name'].value_counts().to_dict()}")

# Parse labels -> (filename, end_sec) -> set of species
segments = {}
with open(DATA_DIR / "train_soundscapes_labels.csv") as f:
    for row in csv.DictReader(f):
        fn = row["filename"]
        parts = row["end"].split(":")
        end_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        key = (fn, end_sec)
        if key not in segments:
            segments[key] = set()
        for sp in str(row["primary_label"]).split(";"):
            segments[key].add(sp.strip())

labeled_species = sorted({sp for spp in segments.values() for sp in spp} & set(all_species))
print(f"Segments: {len(segments)}, species covered: {len(labeled_species)}/{len(all_species)}")

Taxonomy: 234 species
  {'Aves': 162, 'Amphibia': 35, 'Insecta': 28, 'Mammalia': 8, 'Reptilia': 1}
Segments: 739, species covered: 75/234


## Load predictions + build aligned matrices

In [3]:
# Load first model to get segment keys
first_slug = list(MODELS.values())[0]
ref = np.load(INFERENCE_DIR / first_slug / "labeled.npz", allow_pickle=True)
seg_keys_arr = ref["seg_keys"]  # (n, 2) object array: (filename, end_sec)
species_arr = ref["species"]  # (234,) sorted species codes

# Build GT matrix aligned to inference segment order
y_true = np.zeros((len(seg_keys_arr), len(all_species)), dtype=np.float32)
for i, (fn, end_sec) in enumerate(seg_keys_arr):
    key = (fn, int(end_sec))
    if key in segments:
        for sp in segments[key]:
            if sp in sp2idx:
                y_true[i, sp2idx[sp]] = 1.0

# Load predictions
y_pred = {}
for name, slug in MODELS.items():
    data = np.load(INFERENCE_DIR / slug / "labeled.npz", allow_pickle=True)
    y_pred[name] = data["predictions"]
    print(f"{name}: {y_pred[name].shape}")

y_pred["Ensemble"] = np.mean([y_pred[n] for n in MODELS], axis=0)
print(f"\nSegments: {len(seg_keys_arr)}, GT mean labels/seg: {y_true.sum(1).mean():.1f}")

ProtoCLR: (792, 234)
ConvNeXt: (792, 234)
EAT: (792, 234)

Segments: 792, GT mean labels/seg: 3.9


## Per-species AUROC

In [4]:
results = []
for sp_idx, sp in enumerate(all_species):
    col = y_true[:, sp_idx]
    n_pos = int(col.sum())
    if n_pos == 0 or n_pos == len(col):
        continue
    row = {"species": sp, "n_pos": n_pos, "taxon": sp2taxon.get(sp, "Unknown")}
    for m in list(MODELS.keys()) + ["Ensemble"]:
        row[f"auroc_{m}"] = roc_auc_score(col, y_pred[m][:, sp_idx])
    results.append(row)

results_df = pd.DataFrame(results).sort_values("auroc_Ensemble", ascending=True)
print(f"Species with AUROC computable: {len(results_df)}/{len(labeled_species)}")
print(f"\nMacro AUROC ({len(results_df)} species):")
for m in list(MODELS.keys()) + ["Ensemble"]:
    print(f"  {m}: {results_df[f'auroc_{m}'].mean():.4f}")

Species with AUROC computable: 75/75

Macro AUROC (75 species):
  ProtoCLR: 0.8543
  ConvNeXt: 0.8224
  EAT: 0.7769
  Ensemble: 0.8772


## Per-taxon breakdown

In [ ]:
for taxon in sorted(results_df["taxon"].unique()):
    sub = results_df[results_df["taxon"] == taxon]
    print(f"\n{taxon} ({len(sub)} species):")
    for m in list(MODELS.keys()) + ["Ensemble"]:
        col = f"auroc_{m}"
        print(f"  {m}: {sub[col].mean():.4f}  (min={sub[col].min():.3f}, max={sub[col].max():.3f})")


Amphibia (17 species):
  ProtoCLR: 0.8443  (min=0.179, max=0.991)
  ConvNeXt: 0.8202  (min=0.499, max=0.972)
  EAT: 0.8340  (min=0.574, max=0.975)
  Ensemble: 0.8635  (min=0.602, max=0.975)

Aves (28 species):
  ProtoCLR: 0.8542  (min=0.578, max=0.999)
  ConvNeXt: 0.8088  (min=0.063, max=1.000)
  EAT: 0.8226  (min=0.193, max=0.999)
  Ensemble: 0.8993  (min=0.558, max=1.000)

Insecta (25 species):
  ProtoCLR: 0.8555  (min=0.347, max=0.998)
  ConvNeXt: 0.8480  (min=0.519, max=0.991)
  EAT: 0.6732  (min=0.215, max=0.986)
  Ensemble: 0.8574  (min=0.470, max=0.999)

Mammalia (4 species):
  ProtoCLR: 0.8994  (min=0.702, max=1.000)
  ConvNeXt: 0.8692  (min=0.730, max=1.000)
  EAT: 0.9131  (min=0.816, max=1.000)
  Ensemble: 0.9272  (min=0.885, max=1.000)

Reptilia (1 species):
  ProtoCLR: 0.8163  (min=0.816, max=0.816)
  ConvNeXt: 0.4115  (min=0.411, max=0.411)
  EAT: 0.5782  (min=0.578, max=0.578)
  Ensemble: 0.7850  (min=0.785, max=0.785)


## Worst and best species

In [6]:
display_cols = [
    "species",
    "taxon",
    "n_pos",
    "auroc_Ensemble",
    "auroc_ProtoCLR",
    "auroc_ConvNeXt",
    "auroc_EAT",
]

print("=== 15 WORST species (Ensemble AUROC) ===")
print(results_df.head(15)[display_cols].to_string(index=False))

print("\n=== 15 BEST species (Ensemble AUROC) ===")
print(results_df.tail(15)[display_cols].to_string(index=False))

=== 15 WORST species (Ensemble AUROC) ===
   species    taxon  n_pos  auroc_Ensemble  auroc_ProtoCLR  auroc_ConvNeXt  auroc_EAT
47158son06  Insecta     18        0.469710        0.674490        0.753661   0.214686
    trsowl     Aves     26        0.557994        0.578279        0.640289   0.567182
     65377 Amphibia      9        0.601816        0.178658        0.781042   0.640414
   1491113 Amphibia     79        0.610560        0.634474        0.499157   0.736769
47158son04  Insecta     17        0.659583        0.823226        0.518558   0.486148
    limpki     Aves      3        0.674694        0.784537        0.236164   0.642163
47158son10  Insecta     33        0.702639        0.653012        0.790314   0.468000
     67107 Amphibia      9        0.718887        0.837094        0.669647   0.574003
47158son15  Insecta     12        0.724145        0.868376        0.805342   0.599252
   strher2     Aves      2        0.729747        0.829114        0.063291   0.494937
     67252 A

## Species coverage gap

In [7]:
uncovered = sorted(set(all_species) - set(labeled_species))
uncovered_taxa = pd.Series([sp2taxon.get(sp, "Unknown") for sp in uncovered]).value_counts()
print(f"Species NOT in labeled soundscapes: {len(uncovered)}/234")
print(f"  {uncovered_taxa.to_dict()}")

labeled_taxa = pd.Series([sp2taxon.get(sp, "Unknown") for sp in labeled_species]).value_counts()
print(f"\nSpecies IN labeled soundscapes: {len(labeled_species)}/234")
print(f"  {labeled_taxa.to_dict()}")

Species NOT in labeled soundscapes: 159/234
  {'Aves': 134, 'Amphibia': 18, 'Mammalia': 4, 'Insecta': 3}

Species IN labeled soundscapes: 75/234
  {'Aves': 28, 'Insecta': 25, 'Amphibia': 17, 'Mammalia': 4, 'Reptilia': 1}


## Confidence: positive vs negative segments

In [8]:
sp_idxs = [sp2idx[sp] for sp in labeled_species]
for m in list(MODELS.keys()) + ["Ensemble"]:
    p = y_pred[m][:, sp_idxs]
    gt = y_true[:, sp_idxs]
    pos_vals = p[gt == 1]
    neg_vals = p[gt == 0]
    print(
        f"{m}: pos mean={pos_vals.mean():.4f} median={np.median(pos_vals):.4f}  "
        f"neg mean={neg_vals.mean():.4f}  "
        f"sep={pos_vals.mean() / (neg_vals.mean() + 1e-8):.0f}x"
    )

ProtoCLR: pos mean=0.2967 median=0.1400  neg mean=0.0082  sep=36x
ConvNeXt: pos mean=0.2251 median=0.0501  neg mean=0.0063  sep=36x
EAT: pos mean=0.3536 median=0.0748  neg mean=0.0093  sep=38x
Ensemble: pos mean=0.2918 median=0.1871  neg mean=0.0079  sep=37x


## Mel frequency coverage

In [9]:
print("Model frequency coverage:")
for name, slug in MODELS.items():
    with open(WEIGHTS_DIR / slug / "mel_config.json") as f:
        cfg = json.load(f)
    fmax = cfg.get("fmax") or cfg["sr"] // 2
    print(f"  {name}: sr={cfg['sr']}Hz, fmax={fmax}Hz")

Model frequency coverage:
  ProtoCLR: sr=16000Hz, fmax=8000Hz
  ConvNeXt: sr=32000Hz, fmax=16000Hz
  EAT: sr=16000Hz, fmax=8000Hz


## Ensemble vs best single model

In [10]:
results_df["best_single"] = results_df[[f"auroc_{n}" for n in MODELS]].max(axis=1)
results_df["ensemble_gain"] = results_df["auroc_Ensemble"] - results_df["best_single"]

helped = (results_df["ensemble_gain"] > 0.01).sum()
hurt = (results_df["ensemble_gain"] < -0.01).sum()
print(
    f"Ensemble helps (>0.01): {helped}, "
    f"hurts (<-0.01): {hurt}, "
    f"neutral: {len(results_df) - helped - hurt}"
)
print(f"Mean ensemble gain: {results_df['ensemble_gain'].mean():.4f}")

Ensemble helps (>0.01): 8, hurts (<-0.01): 39, neutral: 28
Mean ensemble gain: -0.0395


## Summary

In [11]:
n_files = len({k[0] for k in seg_keys_arr})
print("=" * 60)
print("ERROR ANALYSIS SUMMARY")
print("=" * 60)
print(f"Segments: {len(seg_keys_arr)}, Files: {n_files}")
print(f"Species evaluable: {len(results_df)}/{len(all_species)}")
print()
print("Macro AUROC:")
for m in list(MODELS.keys()) + ["Ensemble"]:
    print(f"  {m}: {results_df[f'auroc_{m}'].mean():.4f}")
print()
print("Per-taxon (Ensemble):")
for taxon in sorted(results_df["taxon"].unique()):
    sub = results_df[results_df["taxon"] == taxon]
    print(f"  {taxon} ({len(sub)} sp): {sub['auroc_Ensemble'].mean():.4f}")
print()
print(f"Not evaluable: {len(uncovered)} species ({uncovered_taxa.to_dict()})")

ERROR ANALYSIS SUMMARY
Segments: 792, Files: 66
Species evaluable: 75/234

Macro AUROC:
  ProtoCLR: 0.8543
  ConvNeXt: 0.8224
  EAT: 0.7769
  Ensemble: 0.8772

Per-taxon (Ensemble):
  Amphibia (17 sp): 0.8635
  Aves (28 sp): 0.8993
  Insecta (25 sp): 0.8574
  Mammalia (4 sp): 0.9272
  Reptilia (1 sp): 0.7850

Not evaluable: 159 species ({'Aves': 134, 'Amphibia': 18, 'Mammalia': 4, 'Insecta': 3})
